# Telco Customer Churn — Data Cleaning & Feature Engineering

**Project:** Telco Customer Churn Prediction
**Stage:** Data Preprocessing & Feature Engineering (CLV)

This notebook loads the raw `CustomerChurn.xlsx` dataset, inspects and cleans it, engineers the
**Customer Lifetime Value (CLV)** feature, and exports a cleaned dataset ready for EDA and
modeling.

**Steps covered:**
1. Load the raw dataset
2. Initial data overview
3. Data quality checks (nulls, duplicates, whitespace, data types)
4. Data cleaning
5. Feature engineering — `CLV = Monthly Charges × Tenure`
6. Final validation
7. Export cleaned dataset


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 2. Load the Raw Dataset

In [2]:
RAW_PATH = "CustomerChurn.xlsx"

df = pd.read_excel(RAW_PATH)
print("Shape:", df.shape)
df.head()


Shape: (7043, 21)


,LoyaltyID,Customer ID,Senior Citizen,Partner,Dependents,Tenure,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn
0,318537,7590-VHVEG,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,152148,5575-GNVDE,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,326527,3668-QPYBK,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,845894,7795-CFOCW,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,503388,9237-HQITU,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Initial Data Overview

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   LoyaltyID          7043 non-null   int64  
 1   Customer ID        7043 non-null   str    
 2   Senior Citizen     7043 non-null   str    
 3   Partner            7043 non-null   str    
 4   Dependents         7043 non-null   str    
 5   Tenure             7043 non-null   int64  
 6   Phone Service      7043 non-null   str    
 7   Multiple Lines     7043 non-null   str    
 8   Internet Service   7043 non-null   str    
 9   Online Security    7043 non-null   str    
 10  Online Backup      7043 non-null   str    
 11  Device Protection  7043 non-null   str    
 12  Tech Support       7043 non-null   str    
 13  Streaming TV       7043 non-null   str    
 14  Streaming Movies   7043 non-null   str    
 15  Contract           7043 non-null   str    
 16  Paperless Billing  7043 non-null   

In [4]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
LoyaltyID,7043.0,NaN,NaN,NaN,550382.651001,260776.11869,100346.0,323604.5,548704.0,776869.0,999912.0
Customer ID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Senior Citizen,7043,2,No,5901,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
Phone Service,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Multiple Lines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Internet Service,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Online Security,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Data Quality Checks

Before cleaning, we check for missing values, duplicate records, whitespace issues in text
columns, and inconsistent data types.

In [5]:
# Missing values per column
df.isnull().sum()


LoyaltyID            0
Customer ID          0
Senior Citizen       0
Partner              0
Dependents           0
Tenure               0
Phone Service        0
Multiple Lines       0
Internet Service     0
Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
Contract             0
Paperless Billing    0
Payment Method       0
Monthly Charges      0
Total Charges        0
Churn                0
dtype: int64

In [6]:
# Exact duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

# Duplicate Customer IDs (should be unique)
print("Duplicate Customer IDs:", df['Customer ID'].duplicated().sum())

# Duplicate LoyaltyIDs (flagged as an identifier, checked for uniqueness)
dup_loyalty = df['LoyaltyID'].duplicated().sum()
print("Duplicate LoyaltyIDs:", dup_loyalty)


Exact duplicate rows: 0
Duplicate Customer IDs: 0
Duplicate LoyaltyIDs: 22


In [7]:
# 'Total Charges' is read as text — check for blank/whitespace values that block numeric conversion
tc_stripped = df['Total Charges'].astype(str).str.strip()
blank_mask = tc_stripped == ""
print("Blank 'Total Charges' values:", blank_mask.sum())
df.loc[blank_mask, ['Customer ID', 'Tenure', 'Monthly Charges', 'Total Charges']]


Blank 'Total Charges' values: 11


,Customer ID,Tenure,Monthly Charges,Total Charges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


In [8]:
# Check every text column for leading/trailing whitespace
text_cols = df.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    vals = df[c].astype(str)
    ws_count = (vals != vals.str.strip()).sum()
    if ws_count:
        print(f"{c}: {ws_count} values with extra whitespace")
print("Whitespace check complete.")


Total Charges: 11 values with extra whitespace
Whitespace check complete.


## 5. Data Cleaning

Based on the checks above, we apply the following cleaning steps:

1. Strip leading/trailing whitespace from all text columns.
2. Convert `Total Charges` to numeric — blank values (found only for customers with `Tenure = 0`,
   i.e. brand-new customers not yet billed) are set to `0`.
3. Remove exact duplicate rows, if any.
4. Correct data types (`Tenure` → int, `Monthly Charges` / `Total Charges` → float).

**Note on `LoyaltyID`:** a small number of `LoyaltyID` values repeat across two different,
otherwise-valid `Customer ID` records. Since `Customer ID` (the true unique key) has no
duplicates and every other field differs between these rows, they are kept as genuine, distinct
customers — this is flagged as a data quality note rather than treated as a duplicate record to
remove.

In [9]:
df_clean = df.copy()

# 5.1 Strip whitespace from all text columns
text_cols = df_clean.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    df_clean[c] = df_clean[c].astype(str).str.strip()

print("Whitespace stripped from:", list(text_cols))


Whitespace stripped from: ['Customer ID', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Total Charges', 'Churn']


In [10]:
# 5.2 Fix 'Total Charges': blank -> NaN -> numeric
df_clean['Total Charges'] = df_clean['Total Charges'].replace("", np.nan)
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce')

missing_before = df_clean['Total Charges'].isna().sum()
print("Missing 'Total Charges' before fill:", missing_before)

# New customers (Tenure = 0) have not been billed yet -> Total Charges = 0
zero_tenure_mask = df_clean['Total Charges'].isna() & (df_clean['Tenure'] == 0)
df_clean.loc[zero_tenure_mask, 'Total Charges'] = 0.0

# Safety net for any other missing values: estimate from Monthly Charges x Tenure
df_clean['Total Charges'] = df_clean['Total Charges'].fillna(
    df_clean['Monthly Charges'] * df_clean['Tenure']
)

print("Missing 'Total Charges' after fill:", df_clean['Total Charges'].isna().sum())


Missing 'Total Charges' before fill: 11
Missing 'Total Charges' after fill: 0


In [11]:
# 5.3 Remove exact duplicate rows (if any)
before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Duplicate rows removed: {before_rows - len(df_clean)}")


Duplicate rows removed: 0


In [12]:
# 5.4 Correct data types
df_clean['Senior Citizen'] = df_clean['Senior Citizen'].astype(str).str.strip()
df_clean['Tenure'] = df_clean['Tenure'].astype(int)
df_clean['Monthly Charges'] = df_clean['Monthly Charges'].astype(float)
df_clean['Total Charges'] = df_clean['Total Charges'].astype(float)

df_clean.dtypes


LoyaltyID              int64
Customer ID              str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure                 int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges        float64
Churn                    str
dtype: object

## 6. Feature Engineering — Customer Lifetime Value (CLV)

Per the project's Business Requirement Document, CLV is calculated as:

$$\text{CLV} = \text{Monthly Charges} \times \text{Tenure}$$


In [13]:
df_clean['CLV'] = (df_clean['Monthly Charges'] * df_clean['Tenure']).round(2)

df_clean[['Customer ID', 'Monthly Charges', 'Tenure', 'Total Charges', 'CLV']].head(10)


,Customer ID,Monthly Charges,Tenure,Total Charges,CLV
0,7590-VHVEG,29.85,1,29.85,29.85
1,5575-GNVDE,56.95,34,1889.50,1936.30
2,3668-QPYBK,53.85,2,108.15,107.70
3,7795-CFOCW,42.30,45,1840.75,1903.50
4,9237-HQITU,70.70,2,151.65,141.40
5,9305-CDSKC,99.65,8,820.50,797.20
6,1452-KIOVK,89.10,22,1949.40,1960.20
7,6713-OKOMC,29.75,10,301.90,297.50
8,7892-POOKP,104.80,28,3046.05,2934.40
9,6388-TABGU,56.15,62,3487.95,3481.30


In [14]:
df_clean['CLV'].describe()


count    7043.000000
mean     2279.581350
std      2264.729447
min         0.000000
25%       394.000000
50%      1393.600000
75%      3786.100000
max      8550.000000
Name: CLV, dtype: float64

## 7. Final Validation

In [15]:
print("Final shape:", df_clean.shape)
print()
null_counts = df_clean.isnull().sum()
remaining_nulls = null_counts[null_counts > 0]
print("Remaining nulls:")
print(remaining_nulls if not remaining_nulls.empty else "None")
print()
print("Remaining duplicate rows:", df_clean.duplicated().sum())
df_clean.head()


Final shape: (7043, 22)

Remaining nulls:
None



Remaining duplicate rows:

 0


,LoyaltyID,Customer ID,Senior Citizen,Partner,Dependents,Tenure,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn,CLV
0,318537,7590-VHVEG,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.85
1,152148,5575-GNVDE,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,1936.30
2,326527,3668-QPYBK,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,107.70
3,845894,7795-CFOCW,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,1903.50
4,503388,9237-HQITU,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,141.40


## 8. Export Cleaned Dataset

In [16]:
OUT_XLSX = "Cleaned_CustomerChurn.xlsx"
OUT_CSV = "Cleaned_CustomerChurn.csv"

df_clean.to_excel(OUT_XLSX, index=False)
df_clean.to_csv(OUT_CSV, index=False)

print(f"Saved cleaned dataset -> {OUT_XLSX}, {OUT_CSV}")
print("Final columns:", list(df_clean.columns))


Saved cleaned dataset -> Cleaned_CustomerChurn.xlsx, Cleaned_CustomerChurn.csv
Final columns: ['LoyaltyID', 'Customer ID', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn', 'CLV']
